# Промпт #25 — Self-Consistency

**Техника:** Self-Consistency  
**Задача:** Запустить один промпт 5 раз и выбрать ответ большинством голосов  
**Сложность:** ⭐⭐⭐☆☆

In [1]:
import sys
sys.path.append('..')
from config import get_completion
from collections import Counter

In [2]:

prompt = """Q: There are 15 trees in the grove. Grove workers will plant trees today. 
After they are done, there will be 21 trees. How many trees did they plant?
A: We start with 15 trees. Later we have 21 trees. The difference must be the number 
of trees they planted. So, they must have planted 21 - 15 = 6 trees. The answer is 6.

Q: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are there?
A: There are 3 cars already. 2 more arrive. Now there are 3 + 2 = 5 cars. The answer is 5.

Q: When I was 6 my sister was half my age. Now I'm 70, how old is my sister?
A:"""

# Запускаем 5 раз с высокой температурой
answers = []
for i in range(5):
    response = get_completion(prompt, temperature=0.9)
    print(f"Запуск {i+1}: {response[:100]}")
    answers.append(response)

# Голосование — ищем "67" или "35" в каждом ответе
votes = []
for answer in answers:
    if "67" in answer:
        votes.append("67")
    elif "35" in answer:
        votes.append("35")
    else:
        votes.append("другой")

print("\n=== ГОЛОСОВАНИЕ ===")
print(f"Все ответы: {votes}")
counter = Counter(votes)
winner = counter.most_common(1)[0][0]
print(f"Победитель: {winner} ({counter[winner]} из 5 запусков)")

Запуск 1: When you were 6, your sister was half your age, which means she was 6 / 2 = 3 years old. 

The diffe
Запуск 2: To find the sister's age, we first need to determine her age when you were 6. Since she was half you
Запуск 3: When you were 6, your sister was half your age, which means she was 6 / 2 = 3 years old. 

The diffe
Запуск 4: When you were 6, your sister was half your age, which is 6 / 2 = 3 years old. 

The difference in ag
Запуск 5: When you were 6, your sister was half your age, which means she was 6 / 2 = 3 years old. 

The diffe

=== ГОЛОСОВАНИЕ ===
Все ответы: ['67', '67', '67', '67', '67']
Победитель: 67 (5 из 5 запусков)


## Оценка: 5/5

## Инсайт
Self-consistency сработала чисто — все 5 запусков дали правильный ответ 67.

Few-shot примеры с пошаговым рассуждением направили модель правильно:
она увидела паттерн "сначала найди разницу в возрасте, потом применяй" 
и повторила его во всех запусках.

Интересно что без few-shot CoT (как в статье https://prompt-engineering-guide.vercel.app/ru/techniques/consistency) модель даёт 35 — 
интуитивный неверный ответ. Few-shot примеры полностью исправили это.

В этом случае голосование было избыточным — все 5 одинаковые.
Self-consistency реально нужна когда модель нестабильна и даёт
разные ответы от запуска к запуску. Тогда большинство голосов
отфильтровывает случайные ошибки.

Главный вывод: few-shot CoT + self-consistency = максимальная надёжность
для задач где точность критична. Цена — в 5 раз больше токенов.